In [3]:
from datasets import load_dataset

In [4]:
dataset = load_dataset('imdb')
small_train = dataset['train'].shuffle(seed=42).select([i for i in list(range(1000))])
small_test = dataset['test'].shuffle(seed=42).select([i for i in list(range(300))])

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

def preprocess_function(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=256)

tokenized_train = small_train.map(preprocess_function, batched=True)
tokenized_test = small_train.map(preprocess_function, batched=True)


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [6]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir = './results',
    num_train_epochs = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size = 16,
    logging_dir = './logs',
    logging_steps = 10,
    report_to = 'none'
)

trainer = Trainer(
    model = model, 
    args = training_args,
    train_dataset = tokenized_train,
    eval_dataset = tokenized_test
)

trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Step,Training Loss
10,0.706157
20,0.641483
30,0.570729
40,0.431295
50,0.320944
60,0.407655
70,0.430210
80,0.246117
90,0.222486
100,0.302489


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=189, training_loss=0.2892913842012012, metrics={'train_runtime': 68.482, 'train_samples_per_second': 43.807, 'train_steps_per_second': 2.76, 'total_flos': 198701097984000.0, 'train_loss': 0.2892913842012012, 'epoch': 3.0})

In [8]:
trainer.evaluate()

{'eval_loss': 0.0588100366294384,
 'eval_runtime': 7.0874,
 'eval_samples_per_second': 141.096,
 'eval_steps_per_second': 8.889,
 'epoch': 3.0}

In [9]:
texts =[ 
    "I really enjoyed this movie!",
    "This was a terrible film.",
    "The plot was predictable and boring.",
]

In [10]:
import torch
import torch.nn.functional as F

In [12]:
inputs = tokenizer(texts, padding = True, truncation = True, return_tensors = 'pt', max_length = 256).to('cuda')

with torch.no_grad():
    outputs = model(**inputs)
    probs = F.softmax(outputs.logits, dim = 1)
    predictions = torch.argmax(probs, dim = 1)

label_map = {0: 'negative', 1: 'positive'}

for text, pred, prob in zip(texts, predictions, probs):
    print(text, label_map[pred.item()], prob[pred.item()].item())

I really enjoyed this movie! positive 0.9842051863670349
This was a terrible film. negative 0.9846981167793274
The plot was predictable and boring. negative 0.9893621802330017


In [13]:
import torch 

def predict_sentiment(text):
    inputs = tokenizer(text, padding = True, truncation = True, return_tensors = 'pt', max_length = 256).to('cuda')

    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim = 1)
        predictions = torch.argmax(probs, dim = 1).item()
    
    return "positive" if predictions == 1 else "negative"

In [14]:
import gradio as gr
interface = gr.Interface(
    fn = predict_sentiment,
    inputs = "text",
    outputs = "text",
    title = "Analyzing sentiment"
)

interface.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2af71f359104f52a60.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [15]:
torch.save(model.state_dict(), 'model.pth')

In [17]:
model.load_state_dict(torch.load('model.pth'))
model.eval()

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
